In [17]:
import numpy as np 
import pandas as pd

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AffinityPropagation
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
import plotly as py
import plotly.graph_objs as go
import os
py.offline.init_notebook_mode(connected = True)
#print(os.listdir("../input"))
import datetime as dt
import missingno as msno
plt.rcParams['figure.dpi'] = 140

In [19]:
file_path = r'C:\Users\jayar\Downloads\project\Netflix-Content-Strat\Week-1-task\netflix_titles (1).csv'

df = pd.read_csv(file_path)
df.head(3)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...


In [20]:
print(f"Number of rows: {len(df)}")

Number of rows: 8807


In [21]:
# count missing values in each column
null_counts = df.isnull().sum()
print(null_counts)

show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


In [22]:
# Explicit null handling per request: fill specific categorical columns,
# drop remaining low-count nulls, and convert date fields to datetime.
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Not available')
df['country'] = df['country'].fillna('Not available')
df['rating'] = df['rating'].fillna('Not rated')

# Drop rows with nulls in columns that have low null counts
# (as requested: 'rating', 'duration', 'date_added')
df = df.dropna(subset=['rating', 'duration', 'date_added'])
print(f"Dataset shape after dropping nulls in rating/duration/date_added: {df.shape}")
# 1. parse date_added normally
# 2. parse release_year using a year-only format; avoid the integer-nanoseconds issue

# using format='%Y' ensures values like 2019 become 2019-01-01
# instead of being interpreted as nanoseconds since epoch (which gave 1970 dates)
df['date_added']   = pd.to_datetime(df['date_added'],   errors='coerce')
df['release_year'] = pd.to_datetime(df['release_year'], format='%Y', errors='coerce')

# keep full timestamp for release_year; extract year later if needed

# verify
def _check_dtypes():
    print(df[['date_added','release_year']].dtypes)

_check_dtypes()
#   date_added      datetime64[ns]
#   release_year    datetime64[ns]
#   dtype: object

# sample values to confirm correct parsing
print(df[['release_year']].head())

# and df.info() should now report datetime dtype(s) for the columns

# Drop rows where conversion produced NaT (non-date values)
df = df.dropna(subset=['date_added', 'release_year'])
print(f"Dataset shape after ensuring date formats: {df.shape}")

df

Dataset shape after dropping nulls in rating/duration/date_added: (8794, 12)
date_added      datetime64[us]
release_year    datetime64[us]
dtype: object
  release_year
0   2020-01-01
1   2021-01-01
2   2021-01-01
3   2021-01-01
4   2021-01-01
Dataset shape after ensuring date formats: (8706, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Not available,United States,2021-09-25,2020-01-01,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,2021-09-24,2021-01-01,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Not available,2021-09-24,2021-01-01,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,Unknown,Not available,Not available,2021-09-24,2021-01-01,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,2021-09-24,2021-01-01,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,2019-11-20,2007-01-01,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,Unknown,Not available,Not available,2019-07-01,2018-01-01,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,2019-11-01,2009-01-01,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,2020-01-11,2006-01-01,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


In [23]:
# Check for duplicates
print("Duplicate rows:")
print(df.duplicated().sum())

# Drop duplicates, keeping the first occurrence
df = df.drop_duplicates(keep='first')

print(f"\nDataset shape after removing duplicates: {df.shape}")
print(f"Total duplicates removed: {len(df) - df.shape[0]}")


Duplicate rows:
0

Dataset shape after removing duplicates: (8706, 12)
Total duplicates removed: 0


In [24]:
# Check for inconsistent categories in the 'type' column
print("Unique values in 'type' column:")
print(df['type'].unique())
print(f"\nValue counts:\n{df['type'].value_counts()}")

# Standardize the 'type' column (fix inconsistencies like 'Tv Show' vs 'TV Show')
df['type'] = df['type'].str.replace('Tv Show', 'TV Show', case=False)

print(f"\nAfter standardization:")
print(df['type'].value_counts())

# Check for other categorical columns with potential inconsistencies
print("\n\nChecking 'rating' column:")
print(f"Unique ratings: {df['rating'].nunique()}")
print(f"Ratings:\n{df['rating'].value_counts()}")
# Check for inconsistencies in other categorical columns
print("\n\nChecking other categorical columns for inconsistencies:")
for col in ['rating', 'country', 'listed_in']:
    if col in df.columns:
        print(f"\n{col.upper()}:")
        print(f"Unique values: {df[col].nunique()}")
        print(df[col].value_counts().head())

# Standardize whitespace issues (leading/trailing spaces)
df = df.map(lambda x: x.strip() if isinstance(x, str) else x)
print("\n\nData standardization complete!")
print(f"Final dataset shape: {df.shape}")
print("\nSummary of data processing:")
print(f"✓ Null values filled with mean/mode (numerical/categorical)")
print(f"✓ Duplicates removed (0 found)")
print(f"✓ Data standardized and whitespace trimmed")
print(f"✓ No rows removed - all data preserved with imputation")


Unique values in 'type' column:
<StringArray>
['Movie', 'TV Show']
Length: 2, dtype: str

Value counts:
type
Movie      6128
TV Show    2578
Name: count, dtype: int64

After standardization:
type
Movie      6128
TV Show    2578
Name: count, dtype: int64


Checking 'rating' column:
Unique ratings: 15
Ratings:
rating
TV-MA        3183
TV-14        2133
TV-PG         838
R             799
PG-13         490
TV-Y7         330
TV-Y          300
PG            287
TV-G          212
NR             78
G              41
TV-Y7-FV        5
Not rated       4
NC-17           3
UR              3
Name: count, dtype: int64


Checking other categorical columns for inconsistencies:

RATING:
Unique values: 15
rating
TV-MA    3183
TV-14    2133
TV-PG     838
R         799
PG-13     490
Name: count, dtype: int64

COUNTRY:
Unique values: 746
country
United States     2775
India              971
Not available      827
United Kingdom     403
Japan              241
Name: count, dtype: int64

LISTED_IN:
Unique va

In [25]:
df.dtypes

# or, for a slightly more verbose summary:
df.info()

<class 'pandas.DataFrame'>
Index: 8706 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8706 non-null   str           
 1   type          8706 non-null   str           
 2   title         8706 non-null   str           
 3   director      8706 non-null   str           
 4   cast          8706 non-null   str           
 5   country       8706 non-null   str           
 6   date_added    8706 non-null   datetime64[us]
 7   release_year  8706 non-null   datetime64[us]
 8   rating        8706 non-null   str           
 9   duration      8706 non-null   str           
 10  listed_in     8706 non-null   str           
 11  description   8706 non-null   str           
dtypes: datetime64[us](2), str(10)
memory usage: 884.2 KB
